# SASA backend benchmark — Colab

Sweeps the `predict_binding_affinity` pipeline across:

- `cpu` — reference PRODIGY / freesasa
- `jax` — blocked Python-loop dispatch (current default)
- `jax-single` — fully-fused single-pass `@jit`
- `jax-scan` — `jax.lax.scan` over blocks (AlphaFold pattern, plays with `jax.checkpoint`)
- `jax-soft` — differentiable sigmoid-smoothed SASA
- `tinygrad` — single-pass on tinygrad

Plots warm-mean pipeline time (ms) vs atom14 atom count, log-log.

## 1. Clone repo + install (Colab)
Skip this cell when running locally inside the repo.

In [ ]:
import os, sys, pathlib
REPO_URL = "https://github.com/mcale6/protein-affinity-gpu.git"
REPO_DIR = pathlib.Path("/content/protein-affinity-gpu")

if "google.colab" in sys.modules:
    if not REPO_DIR.exists():
        os.system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)
    # JAX with CUDA, tinygrad, freesasa, plus repo deps
    os.system("pip install -q --upgrade 'jax[cuda12]' tinygrad freesasa biopython matplotlib")
    os.system("pip install -q -e .")
else:
    REPO_DIR = pathlib.Path.cwd()
    while REPO_DIR != REPO_DIR.parent and not (REPO_DIR / 'pyproject.toml').exists():
        REPO_DIR = REPO_DIR.parent
    os.chdir(REPO_DIR)

sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR))
print("repo root:", REPO_DIR)
import jax
print("jax backend:", jax.default_backend(), "devices:", jax.devices())

## 2. Fetch the Kahraman 2013 T3 fixtures (16 binary complexes, 2k–7k atoms)
The repo ships `benchmarks/run_kahraman_compare.sh` for this; we just download what's missing.

In [ ]:
import csv, urllib.request

MANIFEST = REPO_DIR / 'benchmarks/datasets/kahraman_2013_t3.tsv'
STRUCT_DIR = REPO_DIR / 'benchmarks/downloads/kahraman_2013_t3'
STRUCT_DIR.mkdir(parents=True, exist_ok=True)

with open(MANIFEST) as f:
    next(f)
    pdb_ids = [row.split('\t')[0].upper() for row in f if row.strip()]

for pid in pdb_ids:
    out = STRUCT_DIR / f"{pid}.pdb"
    if out.exists():
        continue
    url = f"https://files.rcsb.org/download/{pid}.pdb"
    print(f"fetch {pid}")
    urllib.request.urlretrieve(url, out)
print(f"have {len(list(STRUCT_DIR.glob('*.pdb')))} / {len(pdb_ids)} structures")

## 3. Run the sweep
Reuses `benchmarks/benchmark.run_benchmark` directly so the notebook stays in sync with the CLI.

On a T4/L4: expect ~2–5 min total. On A100/H100: under a minute. JAX recompiles per N — the first call to each (target, N) eats the compile.

In [ ]:
from pathlib import Path
from benchmarks.benchmark import run_benchmark

OUT_DIR = REPO_DIR / 'benchmarks/output/colab_sweep'
TARGETS = ('cpu', 'jax', 'jax-single', 'jax-scan', 'jax-soft', 'tinygrad')

json_path, report = run_benchmark(
    input_path=None,
    output_dir=OUT_DIR,
    repeats=2,
    targets=TARGETS,
    sphere_points=100,
    manifest=MANIFEST,
    structures_dir=STRUCT_DIR,
)
print('wrote', json_path)
print('rows:', sum(1 for r in report['results'] if r.get('status') == 'ok'))

## 4. Plot — time vs N atoms, one line per backend

In [ ]:
import matplotlib.pyplot as plt

MARKERS = {'cpu': 'x', 'jax': 'o', 'jax-single': 'v', 'jax-scan': 'P',
           'jax-soft': 's', 'tinygrad': 'D'}
COLORS  = {'cpu': '#2ca02c', 'jax': '#1f77b4', 'jax-single': '#9467bd',
           'jax-scan': '#8c564b', 'jax-soft': '#aec7e8', 'tinygrad': '#ff7f0e'}

by_target = {}
for r in report['results']:
    if r.get('status') != 'ok' or r.get('n_atoms') is None:
        continue
    by_target.setdefault(r['target'], []).append((int(r['n_atoms']),
                                                   float(r['warm_mean_seconds']) * 1000.0))

fig, ax = plt.subplots(figsize=(9, 6))
for target in TARGETS:
    pts = sorted(by_target.get(target, []))
    if not pts:
        continue
    xs, ys = zip(*pts)
    ax.plot(xs, ys, marker=MARKERS.get(target, '.'),
            color=COLORS.get(target), linewidth=1.6, markersize=8, label=target)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('atom14 atoms (non-padding)')
ax.set_ylabel('warm-mean pipeline time (ms)')
ax.set_title(f"predict_binding_affinity — JAX backend: {__import__('jax').default_backend()}")
ax.grid(True, which='both', ls=':', alpha=0.4)
ax.legend(frameon=False)
fig.tight_layout()
out_png = OUT_DIR / 'time_vs_atoms.png'
fig.savefig(out_png, dpi=150)
print('saved', out_png)
plt.show()

## 5. Tabular summary

In [ ]:
import pandas as pd
rows = []
for r in report['results']:
    if r.get('status') != 'ok':
        continue
    rows.append({
        'pdb': r['structure_id'],
        'N': r['n_atoms'],
        'target': r['target'],
        'cold_s': round(r['cold_time_seconds'], 3),
        'warm_ms': round(r['warm_mean_seconds'] * 1000, 1),
    })
df = pd.DataFrame(rows)
wide = df.pivot_table(index=['pdb', 'N'], columns='target', values='warm_ms').sort_index(level='N')
wide